# LoRA Fine-tune Gemma 4 E2B — Resume-Job Matcher
Runs on free Colab T4 (16 GB). Training ~30-45 min for 2 epochs over 9k examples.

In [1]:
# Cell 1 — Install
# Unsloth gives 2x faster training and 60% less memory vs vanilla transformers+PEFT

# Install Unsloth and explicit quantization dependencies without forcing an old PyTorch version
%pip install --quiet unsloth datasets trl bitsandbytes xformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
# Cell 2 — Upload training data
# Run prepare_finetune_data.py locally first, then upload finetune_data/ here.
# Option A: upload manually via the Colab file browser (left sidebar)
# Option B: mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
# Then set DATA_DIR to wherever you put finetune_data/ in Drive, e.g.:
DATA_DIR = '/content/drive/MyDrive/finetune_data'
#DATA_DIR = 'finetune_data'  # change if using Drive

Mounted at /content/drive


In [ ]:
# Cell 3 — Load model + apply LoRA  (A100-tuned)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e2b-it",
    max_seq_length=3072,
    load_in_4bit=True,
    dtype=torch.bfloat16,   # A100 supports bf16 natively — preferred for Gemma
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)


### Troubleshooting 'normal_kernel_cuda' Error

This error typically arises when there's an issue with the CUDA installation or an incompatibility between PyTorch and Unsloth's optimized kernels.

**Common Solutions:**

1.  **Restart Runtime:** The simplest fix is often to go to `Runtime > Restart runtime` and try running the cells again. This can clear up transient CUDA issues.
2.  **Verify GPU Runtime:** Ensure you have a GPU runtime enabled (`Runtime > Change runtime type > GPU`).
3.  **Check CUDA Version:** Unsloth generally works best with CUDA 12.1. You can check your CUDA version by running `!nvcc --version`.
4.  **Reinstall PyTorch (Specific Version):** Sometimes, a specific PyTorch version resolves the issue. You can try installing PyTorch with CUDA 12.1:
    ```bash
    %pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
    ```
    After running this, **restart the runtime** and re-run all cells.

I will add a cell to check your current CUDA version.

In [10]:
# Cell 4 — Load dataset
import os
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": os.path.join(DATA_DIR, "train.jsonl"),
    "validation": os.path.join(DATA_DIR, "val.jsonl"),
})
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})


In [11]:
import matplotlib.pyplot as plt
import numpy as np

# Let's tokenize a sample of 500 examples to see the actual sequence lengths
sample_size = min(500, len(ds["train"]))
sample_data = ds["train"].select(range(sample_size))

print(f"Calculating token lengths for {sample_size} examples...")
lengths = []
for example in sample_data:
    text = example.get("text")
    if isinstance(text, str) and text.strip():
        lengths.append(len(tokenizer(text)["input_ids"]))

if not lengths:
    print("Warning: No valid text examples found in the sample! Please check your dataset format.")
else:
    # Plot the distribution
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=50, color='skyblue', edgecolor='black')
    plt.title("Distribution of Sequence Lengths in Training Data")
    plt.xlabel("Number of Tokens")
    plt.ylabel("Frequency")

    # Add vertical lines to show max_seq_length thresholds
    plt.axvline(x=1024, color='red', linestyle='--', label='1024 Tokens (Current Max)')
    plt.axvline(x=3072, color='green', linestyle='--', label='3072 Tokens (Original Max)')
    plt.legend()
    plt.show()

    print(f"Shortest sequence: {min(lengths)} tokens")
    print(f"Longest sequence: {max(lengths)} tokens")
    print(f"95% of your data is under: {int(np.percentile(lengths, 95))} tokens")

Calculating token lengths for 500 examples...


NameError: name 'tokenizer' is not defined

In [ ]:
# Cell 5 — Train  (A100-tuned)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=3072,                  # match the model
        per_device_train_batch_size=4,        # A100 can handle this at bf16
        gradient_accumulation_steps=4,        # effective batch = 16
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        bf16=True,                            # was fp16=True (T4-only); A100 prefers bf16
        fp16=False,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=250,
        save_steps=500,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()


In [ ]:
# Cell 6 — Save LoRA adapter
model.save_pretrained("gemma4-e2b-resume-lora")
tokenizer.save_pretrained("gemma4-e2b-resume-lora")
print("Adapter saved to gemma4-e2b-resume-lora/")

In [ ]:
# Cell 7 — Export GGUF for M1 Mac
# Q4_K_M (~1.2 GB): fast, good for chat. Q5_K_M (~1.5 GB): a bit slower, sharper outputs.
# We export both and let you pick on the Mac.

OUT_DIR = "gemma4-e2b-resume-gguf"

for q in ["q4_k_m", "q5_k_m"]:
    print(f"\n=== Exporting {q} ===")
    model.save_pretrained_gguf(
        OUT_DIR,
        tokenizer,
        quantization_method=q,
    )

# List what landed on disk
import os
for f in sorted(os.listdir(OUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")


In [ ]:
# Cell 7b — Package GGUFs for download
# LM Studio just needs the .gguf file(s) — no Modelfile or chat template config required here.
# We zip them so Drive doesn't try to scan a 1+ GB single file.

import os, glob, shutil

ggufs = sorted(glob.glob(f"{OUT_DIR}/*.gguf"))
print("GGUFs to ship:")
for g in ggufs:
    print(f"  {g}  ({os.path.getsize(g)/1e6:.1f} MB)")

archive = shutil.make_archive("gemma4-e2b-resume-gguf", "zip", OUT_DIR)
print(f"\nZipped to {archive} ({os.path.getsize(archive)/1e6:.1f} MB)")

# Copy to Drive so you can grab it from the Mac without keeping the Colab tab open
DRIVE_OUT = "/content/drive/MyDrive/finetune_data"
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy(archive, DRIVE_OUT)
print(f"Copied to {DRIVE_OUT}/{os.path.basename(archive)}")


In [ ]:
# Cell 8 — Quick sanity check (run before downloading)
import json
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

sample_resume = "Software engineer with 3 years Python, REST APIs, AWS."
sample_job = "Seeking senior Python developer. Required: 5+ years, Kubernetes, CI/CD."

prompt = (
    "<start_of_turn>user\n"
    "Analyze the match between this resume and job posting.\n"
    "Return a JSON object with: match_score, experience_level_fit, rationale, "
    "matching_strengths, skill_gaps, ats_keywords_missing, resume_improvements, "
    "recommended_activities.\n\n"
    f"RESUME:\n{sample_resume}\n\nJOB POSTING:\n{sample_job}\n"
    "<end_of_turn>\n<start_of_turn>model\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the model turn
model_response = response.split("<start_of_turn>model")[-1].strip()
print(model_response)

# Verify it parses as JSON
try:
    parsed = json.loads(model_response)
    print("\nJSON valid:", list(parsed.keys()))
except json.JSONDecodeError as e:
    print("JSON parse error:", e)

## Running on your M1 Mac with LM Studio

### 1. Get the GGUF onto the Mac

Pull `gemma4-e2b-resume-gguf.zip` from your Drive and unzip it. You'll see two files:

- `gemma-4-e2b-it.Q4_K_M.gguf` — ~1.2 GB, fastest, good default
- `gemma-4-e2b-it.Q5_K_M.gguf` — ~1.5 GB, slightly sharper, slightly slower

(Filenames may vary depending on Unsloth's export naming — point is, two `.gguf` files.)

### 2. Sideload into LM Studio

LM Studio expects models under a specific folder structure. Easiest way:

1. Open LM Studio → click the **My Models** tab (folder icon in the left rail).
2. Click the **models directory** path at the top to reveal it in Finder.
3. Inside that directory, create folders: `lmstudio-community/gemma-4-e2b-resume/`
4. Drop both `.gguf` files into that folder.
5. Back in LM Studio, hit the refresh icon — your model appears in the list.

(If your LM Studio version has a "Sideload Model" or "Import GGUF" button in the UI, that does the same thing without manual folder-juggling.)

### 3. Configure the chat template

This is the step people skip and then wonder why outputs are gibberish. In LM Studio:

1. Load your model from the **Chat** tab.
2. Open the right-side panel → **Prompt Template** (or **Preset**).
3. Pick the **Gemma** preset if it's listed. If not, set a custom template:

```
<start_of_turn>user
{prompt}<end_of_turn>
<start_of_turn>model
```

4. Add `<end_of_turn>` as a **stop string** so the model doesn't run on past its turn.
5. Set **Temperature 0.1**, **Top P 0.95** to match what you trained against.

### 4. Sanity test

Paste in a sample resume + job posting using the same format your training data used:

```
Analyze the match between this resume and job posting.
Return a JSON object with: match_score, experience_level_fit, ...

RESUME:
<paste>

JOB POSTING:
<paste>
```

If the output is well-formed JSON in the schema you fine-tuned for, you're done. If it's drifting back to generic Gemma behavior, the chat template is probably wrong — re-check the `<start_of_turn>` / `<end_of_turn>` tokens and the stop string.

### Memory & speed on M1

A 2B Q4_K_M model uses ~1.5 GB resident plus ~1 GB KV cache at 3072 ctx. Even an 8 GB base M1 handles it without swap. Expect ~30-50 tok/s on M1 8-core GPU; M1 Pro/Max will hit 60-90.

### If LM Studio refuses to load it

Gemma 4 is recent — older LM Studio builds may not have the architecture in their bundled llama.cpp yet. Update LM Studio first. As a sanity check, try loading the unmodified base model (`unsloth/gemma-4-e2b-it-GGUF` — searchable from inside LM Studio); if that loads, your fine-tune will too.
